In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
import seaborn as sns
import matplotlib.pyplot as plt

In [21]:
# 1. Load Data

df = pd.read_csv('dataset/CAR DETAILS FROM CAR DEKHO.csv')
print("Original Data Shape:", df.shape)
print(df.head())

Original Data Shape: (4340, 8)
                       name  year  selling_price  km_driven    fuel  \
0             Maruti 800 AC  2007          60000      70000  Petrol   
1  Maruti Wagon R LXI Minor  2007         135000      50000  Petrol   
2      Hyundai Verna 1.6 SX  2012         600000     100000  Diesel   
3    Datsun RediGO T Option  2017         250000      46000  Petrol   
4     Honda Amaze VX i-DTEC  2014         450000     141000  Diesel   

  seller_type transmission         owner  
0  Individual       Manual   First Owner  
1  Individual       Manual   First Owner  
2  Individual       Manual   First Owner  
3  Individual       Manual   First Owner  
4  Individual       Manual  Second Owner  


In [22]:
# 2. Basic Info

print("\nData Types:\n", df.dtypes)
print("\nMissing Values:\n", df.isnull().sum())


Data Types:
 name             object
year              int64
selling_price     int64
km_driven         int64
fuel             object
seller_type      object
transmission     object
owner            object
dtype: object

Missing Values:
 name             0
year             0
selling_price    0
km_driven        0
fuel             0
seller_type      0
transmission     0
owner            0
dtype: int64


In [23]:
# 3. Handle Missing Values

num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['object']).columns

# Numerical: Replace with mean
num_imputer = SimpleImputer(strategy='mean')
df[num_cols] = num_imputer.fit_transform(df[num_cols])

# Categorical: Replace with mode
cat_imputer = SimpleImputer(strategy='most_frequent')
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

print("Missing Values After Imputation:\n", df.isnull().sum().sum())

Missing Values After Imputation:
 0


In [24]:
# 4. Encode Categorical Variables

le = LabelEncoder()

for col in cat_cols:
    if df[col].nunique() == 2:
        df[col] = le.fit_transform(df[col])

df = pd.get_dummies(
    df,
    columns=[col for col in cat_cols if df[col].nunique() > 2],
    drop_first=True
)

print("Shape after Encoding:", df.shape)

Shape after Encoding: (4340, 1504)


In [25]:
# 5. Outlier Detection and Treatment (using IQR)

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = np.where(df[col] < lower, lower, df[col])
    df[col] = np.where(df[col] > upper, upper, df[col])

print("Outlier Treatment Completed")

Outlier Treatment Completed


In [44]:
# 6. Feature Scaling

num_cols = df.select_dtypes(include=['int64', 'float64', 'bool']).columns

scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

print("Feature Scaling Completed")

Feature Scaling Completed


In [38]:
#Feature Selection
target_col = 'selling_price'

if target_col in df.columns:
    X = df.drop(target_col, axis=1)
    y = df[target_col]

    bestfeatures = SelectKBest(score_func=f_classif, k=10)
    fit = bestfeatures.fit(X, y)

    selected_features = X.columns[fit.get_support()]
    df = df[selected_features.to_list() + [target_col]]

    print("\nSelected Features:", selected_features.to_list())


Selected Features: ['name_Mercedes-Benz E-Class E250 CDI Elegance', 'name_Nissan Micra Active XV S', 'name_OpelCorsa 1.6Gls', 'name_Renault KWID Climber 1.0 MT Opt BSIV', 'name_Tata Indica Vista Aqua 1.2 Safire BSIV', 'name_Tata Indica Vista Aqua 1.3 Quadrajet ABS BSIV', 'name_Toyota Innova 2.5 E 8 STR', 'name_Volkswagen Jetta 2.0L TDI Highline AT', 'name_Volkswagen Polo 1.0 TSI Highline Plus', 'name_Volkswagen Vento 1.5 TDI Highline AT']


In [43]:
X = df.drop('selling_price', axis=1)
y = df['selling_price']

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print("Train set:", X_train.shape, y_train.shape)
print("Test set:", X_test.shape, y_test.shape)

Train set: (3038, 10) (3038,)
Test set: (1302, 10) (1302,)


In [37]:
# 1. Linear Regression
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

from sklearn.metrics import r2_score
print("R² Score:", r2_score(y_test, y_pred))

R² Score: 0.0007237984481406334


In [36]:
# 2. Decision Tree
from sklearn.tree import DecisionTreeRegressor

model = DecisionTreeRegressor()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

from sklearn.metrics import r2_score
print("R² Score:", r2_score(y_test, y_pred))

R² Score: 0.0007237984481406334


In [31]:
# 3. Random Forest

model = RandomForestRegressor(n_estimators=100)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

from sklearn.metrics import r2_score
print("R² Score:", r2_score(y_test, y_pred))

R² Score: 0.0005657205896493211


In [32]:
# 4. K-Nearest Neighbors

from sklearn.neighbors import KNeighborsRegressor

model = KNeighborsRegressor(n_neighbors=5)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("KNN R² Score:", r2_score(y_test, y_pred))


KNN R² Score: -0.2059202151722619


In [33]:

# 5. Support Vector Regression

from sklearn.svm import SVR

model = SVR()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("SVR R² Score:", r2_score(y_test, y_pred))

SVR R² Score: -0.056748268226171295


In [34]:


# 6. Gradient Boosting

from sklearn.ensemble import GradientBoostingRegressor

model = GradientBoostingRegressor()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Gradient Boosting R² Score:", r2_score(y_test, y_pred))


Gradient Boosting R² Score: 0.0007232713563516402


In [35]:
# 7. Ridge Regression

from sklearn.linear_model import Ridge

model = Ridge(alpha=1.0)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Ridge Regression R² Score:", r2_score(y_test, y_pred))

Ridge Regression R² Score: 0.0004659321358765345
